In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
import cv2
from glob import glob
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
covid_path = "./dataset/COVID/images/"
covid_mask_path = "./dataset/COVID/masks/"
lung_opacity_path = "./dataset/Lung_Opacity/images/"
lung_opacity_mask_path = "./dataset/Lung_Opacity/masks/"
normal_path = "./dataset/Normal/images/"
normal_mask_path = "./dataset/Normal/masks/"
viral_pneumonia_path = "./dataset/Viral_Pneumonia/images/"
viral_pneumonia_mask_path = "./dataset/Viral_Pneumonia/masks/"

In [ ]:
def getData(dataset_path, mask_path, X_shape, limit=1346):
    im_array = []
    mask_array = []

    dataset_files = set(os.listdir(dataset_path)) & set(os.listdir(mask_path))

    for i in tqdm(dataset_files):
        im = cv2.resize(cv2.imread(os.path.join(dataset_path, i)), (X_shape, X_shape))[:, :, 0]
        mask = cv2.resize(cv2.imread(os.path.join(mask_path, i)), (X_shape, X_shape))[:, :, 0]

        im_array.append(im)
        mask_array.append(mask)

        if len(im_array) >= limit:
            break

    return im_array, mask_array

In [ ]:
dim = 256
X_covid, y_covid = getData(covid_path, covid_mask_path, dim)
X_lung_opacity, y_lung_opacity = getData(lung_opacity_path, lung_opacity_mask_path, dim)
X_normal, y_normal = getData(normal_path, normal_mask_path, dim)
X_viral_pneumonia, y_viral_pneumonia = getData(viral_pneumonia_path, viral_pneumonia_mask_path, dim)

# Convert lists to numpy arrays

In [ ]:
X_covid = np.array(X_covid).reshape(len(X_covid), dim, dim, 1)
y_covid = np.array(y_covid).reshape(len(y_covid), dim, dim, 1)
X_lung_opacity = np.array(X_lung_opacity).reshape(len(X_lung_opacity), dim, dim, 1)
y_lung_opacity = np.array(y_lung_opacity).reshape(len(y_lung_opacity), dim, dim, 1)
X_normal = np.array(X_normal).reshape(len(X_normal), dim, dim, 1)
y_normal = np.array(y_normal).reshape(len(y_normal), dim, dim, 1)
X_viral_pneumonia = np.array(X_viral_pneumonia).reshape(len(X_viral_pneumonia), dim, dim, 1)
y_viral_pneumonia = np.array(y_viral_pneumonia).reshape(len(y_viral_pneumonia), dim, dim, 1)


In [ ]:
X_data = np.concatenate((X_covid, X_lung_opacity, X_normal, X_viral_pneumonia), axis=0)
y_data = np.concatenate((y_covid, y_lung_opacity, y_normal, y_viral_pneumonia), axis=0)

In [ ]:
print("X_data shape:", X_data.shape)
print("y_data shape:", y_data.shape)

In [ ]:
class LungSegmentationDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        return image, mask

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
])


In [ ]:
X_data = (X_data - 127.0) / 127.0
y_data = (y_data > 127).astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.1, random_state=2018)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.1, random_state=2018)

# Create datasets and dataloaders
train_dataset = LungSegmentationDataset(X_train, y_train, transform=train_transform)
val_dataset = LungSegmentationDataset(X_val, y_val, transform=val_transform)
test_dataset = LungSegmentationDataset(X_test, y_test, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        
        def CBR2d(in_channels, out_channels, kernel_size, stride, padding, bias=True):
            layers = []
            layers += [nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias)]
            layers += [nn.BatchNorm2d(num_features=out_channels)]
            layers += [nn.ReLU()]
            cbr = nn.Sequential(*layers)
            return cbr
        
        self.enc1_1 = CBR2d(1, 64, 3, 1, 1)
        self.enc1_2 = CBR2d(64, 64, 3, 1, 1)
        
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2_1 = CBR2d(64, 128, 3, 1, 1)
        self.enc2_2 = CBR2d(128, 128, 3, 1, 1)
        
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3_1 = CBR2d(128, 256, 3, 1, 1)
        self.enc3_2 = CBR2d(256, 256, 3, 1, 1)
        
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4_1 = CBR2d(256, 512, 3, 1, 1)
        self.enc4_2 = CBR2d(512, 512, 3, 1, 1)
        
        self.pool4 = nn.MaxPool2d(2)
        
        self.enc5_1 = CBR2d(512, 1024, 3, 1, 1)
        
        self.unpool4 = nn.ConvTranspose2d(1024, 512, 2, 2, 0)
        self.dec4_1 = CBR2d(2*512, 512, 3, 1, 1)
        self.dec4_2 = CBR2d(512, 512, 3, 1, 1)
        
        self.unpool3 = nn.ConvTranspose2d(512, 256, 2, 2, 0)
        self.dec3_1 = CBR2d(2*256, 256, 3, 1, 1)
        self.dec3_2 = CBR2d(256, 256, 3, 1, 1)
        
        self.unpool2 = nn.ConvTranspose2d(256, 128, 2, 2, 0)
        self.dec2_1 = CBR2d(2*128, 128, 3, 1, 1)
        self.dec2_2 = CBR2d(128, 128, 3, 1, 1)
        
        self.unpool1 = nn.ConvTranspose2d(128, 64, 2, 2, 0)
        self.dec1_1 = CBR2d(2*64, 64, 3, 1, 1)
        self.dec1_2 = CBR2d(64, 64, 3, 1, 1)
        
        self.fc = nn.Conv2d(64, out_channels, 1, 1, 0)
        
    def forward(self, x):
        enc1_1 = self.enc1_1(x)
        enc1_2 = self.enc1_2(enc1_1)
        pool1 = self.pool1(enc1_2)
        
        enc2_1 = self.enc2_1(pool1)
        enc2_2 = self.enc2_2(enc2_1)
        pool2 = self.pool2(enc2_2)
        
        enc3_1 = self.enc3_1(pool2)
        enc3_2 = self.enc3_2(enc3_1)
        pool3 = self.pool3(enc3_2)
        
        enc4_1 = self.enc4_1(pool3)
        enc4_2 = self.enc4_2(enc4_1)
        pool4 = self.pool4(enc4_2)
        
        enc5_1 = self.enc5_1(pool4)
        
        unpool4 = self.unpool4(enc5_1)
        concat4 = torch.cat((unpool4, enc4_2), dim=1)
        dec4_1 = self.dec4_1(concat4)
        dec4_2 = self.dec4_2(dec4_1)
        
        unpool3 = self.unpool3(dec4_2)
        concat3 = torch.cat((unpool3, enc3_2), dim=1)
        dec3_1 = self.dec3_1(concat3)
        dec3_2 = self.dec3_2(dec3_1)
        
        unpool2 = self.unpool2(dec3_2)
        concat2 = torch.cat((unpool2, enc2_2), dim=1)
        dec2_1 = self.dec2_1(concat2)
        dec2_2 = self.dec2_2(dec2_1)
        
        unpool1 = self.unpool1(dec2_2)
        concat1 = torch.cat((unpool1, enc1_2), dim=1)
        dec1_1 = self.dec1_1(concat1)
        dec1_2 = self.dec1_2(dec1_1)
        
        x = self.fc(dec1_2)
        
        return x

In [ ]:
model = UNet()

In [ ]:
model = model.to(device)

In [ ]:
def dice_coef(y_true, y_pred):
    smooth = 1.
    y_true_f = y_true.view(-1)
    y_pred_f = y_pred.view(-1)
    intersection = (y_true_f * y_pred_f).sum()
    return (2. * intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, y_true, y_pred):
        return 1 - dice_coef(y_true, y_pred)


In [ ]:
criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training and validation loop
num_epochs = 50
best_val_loss = float('inf')

In [ ]:
train_loss_history = []
val_loss_history = []
train_dice_history = []
val_dice_history = []

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    train_dice = 0
    for images, masks in train_loader:
        images = images.float().cuda()
        masks = masks.float().cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_dice += dice_coef(masks, outputs).item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    train_dice /= len(train_loader.dataset)
    train_loss_history.append(train_loss)
    train_dice_history.append(train_dice)

    model.eval()
    val_loss = 0
    val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.float().cuda()
            masks = masks.float().cuda()
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)
            val_dice += dice_coef(masks, outputs).item() * images.size(0)

    val_loss /= len(val_loader.dataset)
    val_dice /= len(val_loader.dataset)
    val_loss_history.append(val_loss)
    val_dice_history.append(val_dice)

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Train Dice: {train_dice:.4f}, Val Dice: {val_dice:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")

# Load the best model
model.load_state_dict(torch.load("best_model.pth"))

In [ ]:
model.eval()
test_loss = 0
test_dice = 0
with torch.no_grad():
    for images, masks in test_loader:
        images = images.float().cuda()
        masks = masks.float().cuda()
        outputs = model(images)
        loss = criterion(outputs, masks)
        test_loss += loss.item() * images.size(0)
        test_dice += dice_coef(masks, outputs).item() * images.size(0)

test_loss /= len(test_loader.dataset)
test_dice /= len(test_loader.dataset)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss_history, label='Training Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_dice_history, label='Training Dice Coefficient')
plt.plot(val_dice_history, label='Validation Dice Coefficient')
plt.title('Training and Validation Dice Coefficient')
plt.xlabel('Epoch')
plt.ylabel('Dice Coefficient')
plt.legend()

plt.show()